# Week 3 Exercise - Synthetic Dataset Generator

Generate a synthetic dataset using multiple models and prompt variants.
This notebook builds a small Gradio app for dataset creation.

In [2]:
# imports
import os
import json
import random
from dotenv import load_dotenv
import torch
from transformers import pipeline
import pandas as pd
import gradio as gr

In [3]:
# setup
load_dotenv(override=True)

HF_TOKEN = os.getenv("HF_TOKEN")

MODELS = {
    "GPT-2": {"model_id": "gpt2"},
    "DistilGPT2": {"model_id": "distilgpt2"},
    "TinyLlama": {"model_id": "TinyLlama/TinyLlama-1.1B-Chat-v1.0"},
}

PIPELINE_CACHE = {}

def get_pipeline(model_key):
    if model_key in PIPELINE_CACHE:
        return PIPELINE_CACHE[model_key]
    model_id = MODELS[model_key]["model_id"]
    pipe = pipeline(
        "text-generation",
        model=model_id,
        token=HF_TOKEN if HF_TOKEN else None,
        device=0 if torch.cuda.is_available() else -1,
    )
    PIPELINE_CACHE[model_key] = pipe
    return pipe

In [4]:
# prompt variants for diversity
PROMPT_TEMPLATES = [
    "Create one JSON object that follows the schema.",
    "Generate one synthetic record as JSON only.",
    "Return a single JSON object that fits the schema exactly.",
]

SYSTEM_PROMPT = """You are a data generator. Return valid JSON only.
Do not include extra text, markdown, or code fences.
Use keys exactly as provided in the schema."""

TOPICS = [
    "Customer support issues",
    "E-commerce product feedback",
    "Healthcare appointment summaries",
    "Travel booking requests",
    "SaaS bug reports",
]

DATASET_TEMPLATES = {
    "Q&A Pairs": {
        "schema": "question, answer",
        "prompt": "Generate a question and answer about {topic}. Return JSON with keys: question, answer.",
    },
    "Issue Tickets": {
        "schema": "id, issue, severity, resolution",
        "prompt": "Generate a support ticket about {topic}. Return JSON with keys: id, issue, severity, resolution.",
    },
    "Product Reviews": {
        "schema": "id, product, rating, review",
        "prompt": "Generate a product review about {topic}. Return JSON with keys: id, product, rating, review.",
    },
}

In [5]:
def call_model(model_key, prompt, max_new_tokens, temperature, top_p):
    pipe = get_pipeline(model_key)
    output = pipe(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        num_return_sequences=1,
    )
    # Transformers returns the full prompt + continuation
    generated = output[0]["generated_text"]
    return generated[len(prompt):].strip()


def extract_json(text):
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return None
    return text[start : end + 1]


def parse_json(text, fallback_schema):
    snippet = extract_json(text)
    if not snippet:
        return {"error": "invalid_json", "raw": text, **fallback_schema}
    try:
        return json.loads(snippet)
    except json.JSONDecodeError:
        return {"error": "invalid_json", "raw": text, **fallback_schema}

In [6]:
def generate_dataset(topic, schema, n_rows, strategy, single_model, max_new_tokens, temperature, top_p, template_prompt):
    # Build schema dict from comma-separated fields
    fields = [f.strip() for f in schema.split(",") if f.strip()]
    base_schema = {f: "" for f in fields}

    results = []
    model_names = list(MODELS.keys())

    for i in range(n_rows):
        if strategy == "rotate":
            model_choice = model_names[i % len(model_names)]
        else:
            model_choice = single_model

        prompt_variant = random.choice(PROMPT_TEMPLATES)
        template_line = template_prompt.format(topic=topic)
        prompt = (
            f"{SYSTEM_PROMPT}\n"
            f"Topic: {topic}\n"
            f"Schema fields: {', '.join(fields)}\n"
            f"{template_line}\n"
            f"{prompt_variant}"
        )

        raw = call_model(model_choice, prompt, max_new_tokens, temperature, top_p)
        row = parse_json(raw, {**base_schema, "model": model_choice})
        if "model" not in row:
            row["model"] = model_choice
        results.append(row)

    return results

In [7]:
def format_display(rows):
    if not rows:
        return "No rows generated."
    lines = [f"Total rows: {len(rows)}\n"]
    for i, row in enumerate(rows, start=1):
        lines.append(f"Row {i}: {row}")
    return "\n".join(lines)


def export_json(rows):
    if not rows:
        return None
    filename = "synthetic_dataset.json"
    with open(filename, "w") as f:
        json.dump(rows, f, indent=2)
    return filename


def export_csv(rows):
    if not rows:
        return None
    df = pd.DataFrame(rows)
    filename = "synthetic_dataset.csv"
    df.to_csv(filename, index=False)
    return filename


def run_generator(topic, schema, n_rows, strategy, single_model, max_new_tokens, temperature, top_p, template_choice):
    template = DATASET_TEMPLATES[template_choice]
    if not schema.strip():
        schema = template["schema"]
    data = generate_dataset(
        topic,
        schema,
        n_rows,
        strategy,
        single_model,
        max_new_tokens,
        temperature,
        top_p,
        template["prompt"],
    )
    df = pd.DataFrame(data)
    return df, data, format_display(data), export_json(data), export_csv(data)


# Gradio UI
with gr.Blocks(title="Synthetic Dataset Generator", theme=gr.themes.Soft()) as ui:
    gr.Markdown("# Synthetic Dataset Generator\nGenerate structured datasets with multiple HF models and prompt variants.")

    with gr.Row():
        with gr.Column(scale=1):
            topic_input = gr.Dropdown(TOPICS, value=TOPICS[0], label="Dataset topic")
            template_input = gr.Dropdown(list(DATASET_TEMPLATES.keys()), value="Issue Tickets", label="Template")
            schema_input = gr.Textbox(label="Schema fields (comma separated)", value="")
            rows_input = gr.Slider(minimum=1, maximum=50, value=10, step=1, label="Rows")
            strategy_input = gr.Dropdown(["rotate", "single"], value="rotate", label="Model strategy")
            model_input = gr.Dropdown(list(MODELS.keys()), value="GPT-2", label="Single model")
            max_tokens_input = gr.Slider(minimum=32, maximum=256, value=128, step=16, label="Max new tokens")
            temperature_input = gr.Slider(minimum=0.1, maximum=1.2, value=0.7, step=0.1, label="Temperature")
            top_p_input = gr.Slider(minimum=0.5, maximum=1.0, value=0.9, step=0.05, label="Top-p")
            run_btn = gr.Button("Generate", variant="primary")

            json_file = gr.File(label="Download JSON", interactive=False)
            csv_file = gr.File(label="Download CSV", interactive=False)

        with gr.Column(scale=2):
            output_df = gr.Dataframe(label="Generated dataset")
            output_json = gr.JSON(label="Raw JSON")
            output_text = gr.Textbox(label="Preview", lines=14)

    run_btn.click(
        fn=run_generator,
        inputs=[
            topic_input,
            schema_input,
            rows_input,
            strategy_input,
            model_input,
            max_tokens_input,
            temperature_input,
            top_p_input,
            template_input,
        ],
        outputs=[output_df, output_json, output_text, json_file, csv_file],
    )

ui.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
